# 🔍 Fake Review Detection Model

Train a classifier to detect fake/genuine reviews using SVM, Logistic Regression, and Random Forest.

**Algorithms**: SVM (primary), Logistic Regression, Random Forest
**Input**: Review text
**Output**: Fake/Genuine classification


In [ ]:
# Install required libraries
!pip install -q scikit-learn pandas numpy nltk matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import joblib
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

print('✅ All libraries imported successfully')

## 📊 Load Sample Data

You can replace this with your actual dataset. The dataset should have columns: `review`, `label` (0=Genuine, 1=Fake)

In [ ]:
# Sample training data (replace with your actual dataset)
sample_data = {
    'review': [
        'Great product, highly recommend!',  # Genuine
        'Amazing quality, best buy ever',      # Genuine
        'This is a fake review for money',     # Fake
        'BUY NOW LIMITED OFFER!!!',            # Fake
        'Excellent service and fast delivery', # Genuine
        'Paid review, not real opinion',       # Fake
        'Five stars, love it',                 # Genuine
        'Click here to buy more',              # Fake
        'Product works as described',          # Genuine
        'Dont waste your money',               # Genuine
    ],
    'label': [0, 0, 1, 1, 0, 1, 0, 1, 0, 0]  # 0=Genuine, 1=Fake
}

df = pd.DataFrame(sample_data)
print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['label'].value_counts())
print('\nSample data:')
print(df.head())

## 🧹 Data Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and normalize text"""
    if not isinstance(text, str):
        return ''
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters
    import re
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
df['review_clean'] = df['review'].apply(preprocess_text)
print('✅ Preprocessing complete')
print('\nCleaned samples:')
print(df[['review', 'review_clean']].head())

## 🔢 Feature Vectorization (TF-IDF)

In [ ]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['review_clean'])
y = df['label']

print(f'✅ Vectorization complete')
print(f'Feature matrix shape: {X.shape}')
print(f'Number of features: {len(vectorizer.get_feature_names_out())}')
print(f'\nTop 10 features:')
for i, feature in enumerate(vectorizer.get_feature_names_out()[:10]):
    print(f'{i+1}. {feature}')

## 📚 Train-Test Split

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Train-test split complete')
print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')
print(f'\nTraining set class distribution:')
print(pd.Series(y_train).value_counts())

## 🤖 Train Models

In [ ]:
from sklearn.metrics import roc_auc_score

# 1. Support Vector Machine (SVM)
print('Training SVM...')
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_pred)
print(f'✅ SVM Accuracy: {svm_acc:.4f}')

# 2. Logistic Regression
print('\nTraining Logistic Regression...')
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print(f'✅ Logistic Regression Accuracy: {lr_acc:.4f}')

# 3. Random Forest
print('\nTraining Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'✅ Random Forest Accuracy: {rf_acc:.4f}')

## 📊 Model Evaluation

In [ ]:
# Detailed evaluation for best model (SVM)
print('='*60)
print('SUPPORT VECTOR MACHINE (SVM) - PRIMARY MODEL')
print('='*60)
print('\nClassification Report:')
print(classification_report(y_test, svm_pred, target_names=['Genuine', 'Fake']))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, svm_pred)
print(cm)

# Model comparison
print('\n' + '='*60)
print('MODEL COMPARISON')
print('='*60)
models_comparison = pd.DataFrame({
    'Model': ['SVM', 'Logistic Regression', 'Random Forest'],
    'Accuracy': [svm_acc, lr_acc, rf_acc]
})
models_comparison = models_comparison.sort_values('Accuracy', ascending=False)
print(models_comparison.to_string(index=False))

## 💾 Save Models

In [ ]:
# Save the best model (SVM)
joblib.dump(svm_model, 'fake_review_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

print('✅ Models saved successfully!')
print('   - fake_review_model.pkl')
print('   - vectorizer.pkl')
print('\nDownload these files and place them in: backend/models/trained_models/')

## 🧪 Test the Model

In [ ]:
# Test with new reviews
test_reviews = [
    'Excellent product, very satisfied',
    'BUY NOW CLICK HERE LIMITED TIME',
    'Works perfectly, great value for money'
]

print('Testing model on new reviews:')
print('='*60)

for review in test_reviews:
    # Preprocess
    review_clean = preprocess_text(review)
    # Vectorize
    X_new = vectorizer.transform([review_clean])
    # Predict
    pred = svm_model.predict(X_new)[0]
    confidence = max(svm_model.predict_proba(X_new)[0])
    
    label = 'FAKE' if pred == 1 else 'GENUINE'
    print(f'Review: {review}')
    print(f'Prediction: {label} (Confidence: {confidence:.2%})')
    print('-'*60)